In [1]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy as np
import json
from typing import Optional, Dict, Sequence, List
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy
import random


print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0

Torch version:2.7.1+cu126
Cuda version: 12.6
transformers version: 4.56.1


### SFT

- 모델, 데이터셋 구성

In [4]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = AutoTokenizer.from_pretrained(
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,
)

In [5]:
class SFT_dataset(Dataset):
    def __init__(self, list_data_dict: List[Dict], tokenizer: transformers.PreTrainedTokenizer):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }
        prompt_input = PROMPT_DICT["prompt_input"]

        # 소스와 타겟 생성
        sources = [prompt_input.format_map(example) for example in list_data_dict]
        targets = [f"{example['completion']}{tokenizer.eos_token}" for example in list_data_dict]
        examples = [s + t for s, t in zip(sources, targets)]

        # 토크나이징
        sources_tokenized = self._tokenize_fn(sources, tokenizer)
        examples_tokenized = self._tokenize_fn(examples, tokenizer)

        # 레이블 생성 (Source 부분은 -100으로 마스킹)
        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        self.input_ids = input_ids
        self.labels = labels
        logging.warning(f"Loading data done!!: {len(self.labels)}")

    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        input_ids = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            input_ids_lens=input_ids_lens,
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

@dataclass
class DataCollatorForSupervisedDataset(object):
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )


In [6]:
tokenizer.eos_token_id

1

In [7]:
# SFT 학습 전/후 비교를 위한 함수
PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id, # eos_token='</s>'의 ID 사용
    max_new_tokens=128,
    do_sample=True,
    top_k=50,
    temperature=1.0,
    early_stopping=True
)

# generation_args = dict(
#     num_beams=4,
#     repetition_penalty=2.0,
#     no_repeat_ngram_size=4,
#     eos_token_id=1, # \n
#     max_new_tokens=128,
#     do_sample=True,
#     top_k=50,
#     early_stopping=True
# )

import evaluate
from tqdm.auto import tqdm

def evaluate_model(model, tokenizer, test_dataset):
    device = 0 if torch.cuda.is_available() else -1
    generator = transformers.pipeline('text-generation', model=model, tokenizer=tokenizer, device=device)

    # 프롬프트 형식
    prompt_template = PROMPT_DICT['prompt_input']

    predictions = []
    references = []

    print("Generating predictions for evaluation...")
    # tqdm을 사용하여 진행 상황 표시
    for item in tqdm(test_dataset):
        prompt = prompt_template.format_map({'prompt': item['prompt']})

        # 모델 예측 생성
        result = generator(prompt, **generation_args)
        generated_text = result[0]['generated_text']
        response = generated_text.split("### Response(응답):")[1].strip()

        # 예측과 정답을 리스트에 추가
        predictions.append(response)
        references.append(item['completion'])

    print("Calculating scores...")

    # ROUGE 점수 계산
    rouge = evaluate.load('rouge')
    rouge_results = rouge.compute(predictions=predictions, references=references)

    print("\n" + "="*30)
    print("Quantitative Evaluation Results")
    print("="*30)
    print("ROUGE Scores:")
    for key, value in rouge_results.items():
        print(f"  {key}: {value*100:.2f}") # 백분율로 표시
    print("="*30)

    # 질적 평가를 위해 몇 가지 샘플 출력
    print("\nQualitative Samples:")
    for i in range(3): # 3개 샘플만 확인
        print(f"\n--- Sample {i+1} ---")
        print(f"✅ Prompt: {test_dataset[i]['prompt']}")
        print(f"🎯 Ground Truth: {references[i]}")
        print(f"🤖 Generated: {predictions[i]}")
        print("-" * 20)

In [8]:
with open('./data_kochatgpt/kochatgpt_1_SFT.jsonl', "r", encoding='utf-8-sig') as f:
    data = json.load(f)

random.shuffle(data)
train_data = data[:-100]  # 훈련용 list
test_data = data[-100:]   # 평가용 list (훈련과 완벽히 분리)

In [57]:
print("\n" + "="*50)
print("PERFORMANCE EVALUATION (BEFORE SFT)")
print("="*50)

evaluate_model(model, tokenizer, test_data)

Device set to use cuda:0



PERFORMANCE EVALUATION (BEFORE SFT)
Generating predictions for evaluation...


  0%|          | 0/100 [00:00<?, ?it/s]

Calculating scores...

Quantitative Evaluation Results
ROUGE Scores:
  rouge1: 0.00
  rouge2: 0.00
  rougeL: 0.00
  rougeLsum: 0.00

Qualitative Samples:

--- Sample 1 ---
✅ Prompt: 결국 헤어짐이 확실해졌습니다
🎯 Ground Truth: '저는 AI 챗봇으로써 사람처럼 감정을 느끼지는 않기 때문에, 헤어짐이 확실해졌다는 것이 어떤 상황을 말하는 것인지 정확히 파악할 수 없습니다. 그러나 이별이 어려운 시기라면 많은 사람들이 느끼는 감정들을 들어줄 수 있습니다. 이별은 상처를 줄 수도 있지만, 그 경험을 바탕으로 성장할 수 있는 기회가 될 수도 있습니다. 혼자서 처리하기 어려운 감정이나 생각이 있으면, 가족, 친구, 전문가들의 도움을 받는 것도 좋은 방법일 수 있습니다.
🤖 Generated: #Responsibility #selfie #daily #like4like #instagood #cafe #neko #katze #meow #냥스타그램 #catsoninstagram #고양이 #예쁜고양이 #20190710 #미롱_식단 
점심 : 닭가슴살 + 당근샐러드
간식 : 달걀후라이드
공복시간 :
--------------------

--- Sample 2 ---
✅ Prompt: PHEIC가 뭐지
🎯 Ground Truth: 'PHEIC는 공중보건 위기 상황을 나타내는 용어로, "Public Health Emergency of International Concern"의 줄임말입니다. 세계보건기구(WHO)가 발표하는 승인된 지정 용어 중 하나로, 감염병 등 급작스러운 공중보건 위기 상황이 발생한 경우 해당 국가 또는 지역에서 WHO의 협조와 지원을 받을 수 있습니다.
🤖 Generated: #reading #daily #like4like #selfie #instagood #ootd #outfitness #셀카 #셀피 #얼스타

- SFT를 하지 않은 KoGPT2 모델은 rouge 점수는 0이고 답변을 육안으로 봤을 때에도 전혀 상관없는 답변을 하고있다.

In [63]:
train_dataset = SFT_dataset(list_data_dict=train_data, tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print(len(train_dataset))

11900


In [65]:
training_args = transformers.TrainingArguments(
    output_dir="test",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=10,
    prediction_loss_only=True,
    bf16=True,
    logging_steps=100,
    load_best_model_at_end=True,
)
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [66]:
trainer.train()
model.save_pretrained('models/output_1_SFT')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.003900
200,2.811900
300,2.742100
400,2.605100
500,2.332900
600,2.334500
700,2.338600
800,2.201300
900,2.061400
1000,2.079700


In [67]:
print("\n" + "="*50)
print("PERFORMANCE EVALUATION (AFTER SFT)")
print("="*50)

model = AutoModelForCausalLM.from_pretrained('models/output_1_SFT')
evaluate_model(model, tokenizer, test_data)

Device set to use cuda:0



PERFORMANCE EVALUATION (AFTER SFT)
Generating predictions for evaluation...


  0%|          | 0/100 [00:00<?, ?it/s]

Calculating scores...

Quantitative Evaluation Results
ROUGE Scores:
  rouge1: 7.69
  rouge2: 0.00
  rougeL: 7.62
  rougeLsum: 7.55

Qualitative Samples:

--- Sample 1 ---
✅ Prompt: 결국 헤어짐이 확실해졌습니다
🎯 Ground Truth: '저는 AI 챗봇으로써 사람처럼 감정을 느끼지는 않기 때문에, 헤어짐이 확실해졌다는 것이 어떤 상황을 말하는 것인지 정확히 파악할 수 없습니다. 그러나 이별이 어려운 시기라면 많은 사람들이 느끼는 감정들을 들어줄 수 있습니다. 이별은 상처를 줄 수도 있지만, 그 경험을 바탕으로 성장할 수 있는 기회가 될 수도 있습니다. 혼자서 처리하기 어려운 감정이나 생각이 있으면, 가족, 친구, 전문가들의 도움을 받는 것도 좋은 방법일 수 있습니다.
🤖 Generated: '저는 인공지능 어시스턴트이기 때문에 감정을 느끼지는 않습니다. 하지만 이별은 언제나 힘든 일입니다. 그러나 이별을 예상하고 이를 극복하는 방법은 다양할 수 있습니다.\n\n1. 새로운 취미나 관심사를 찾아보세요.\n새로운 취미를 찾아보는 것도 좋은 방법입니다. 예를 들어, 영화를 보거나 책을 읽는 등 새로운 경험을 할 수 있는 활동을 해보세요. 또한, 자신의 생각과 감정을 솔직하게 표현하는 것도 좋은 방법일 수 있습니다.
--------------------

--- Sample 2 ---
✅ Prompt: PHEIC가 뭐지
🎯 Ground Truth: 'PHEIC는 공중보건 위기 상황을 나타내는 용어로, "Public Health Emergency of International Concern"의 줄임말입니다. 세계보건기구(WHO)가 발표하는 승인된 지정 용어 중 하나로, 감염병 등 급작스러운 공중보건 위기 상황이 발생한 경우 해당 국가 또는 지역에서 WHO의 협조와 지원을 받을 수 있습니다.
🤖 Generated: '저는 인공

- SFT를 적용한 KoGPT2는 rouge 점수도 여전히 낮긴 하지만 상당히 개선되었고 육안으로 봤을 때도 그럴듯한 답변을 하는 것을 볼 수 있음

### Reward Model 학습

In [13]:
torch.cuda.empty_cache()

In [13]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

- Top-vs-Rest(최상위 vs 나머지)로 데이터 구성
- GhatGPT와 text-davinci-003, text-ada-001로 답변을 생성한 데이터인데 davinci와 ada는 큰 차이가 없다고 가정

In [15]:
with open('./data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []

for tmp in list_data_dict:
     prompt = tmp['prompt']
     ranking = tmp['ranking']

     for index in range(1, len(ranking)):
         n = ranking[0]
         m = ranking[index]


         data = {
             'prompt': prompt,
             'chosen': tmp['completion_{}'.format(n)],
             'rejected': tmp['completion_{}'.format(m)]
         }

         total_data_ranking2chosen.append(data)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example1: \n%s'%total_data_ranking2chosen[46])
print('data example2: \n%s'%total_data_ranking2chosen[47])

before data num: 10220
after  data num: 20440
data example1: 
{'prompt': '메뉴판 있어요?', 'chosen': '죄송합니다, 제가 가상의 챗봇입니다. 제가 메뉴판을 제공할 수는 없습니다. 하지만, 혹시 궁금한 음식이나 음료가 있다면 알려주시면 그에 맞는 정보를 제공해드릴 수 있습니다.', 'rejected': '네, 여러가지 메뉴가 있습니다. 치킨, 피자, 샐러드, 스파게티, 그리고 다양한 간식과 음료 등이 있습니다.'}
data example2: 
{'prompt': '메뉴판 있어요?', 'chosen': '죄송합니다, 제가 가상의 챗봇입니다. 제가 메뉴판을 제공할 수는 없습니다. 하지만, 혹시 궁금한 음식이나 음료가 있다면 알려주시면 그에 맞는 정보를 제공해드릴 수 있습니다.', 'rejected': '이락으로 잘알면 강의할 지 등 공개한 경개는 높아 가고 한다면 객체개를 위한 개서를 한다가 국토법인 경개를 높게 속한 경개는 일 하나 국토법인 경개를 들여 객체개는 더 국토법인 경개를 높이 한다면 객체개를 토법인 경개를 높이 한다고 국토법인 경개를 한다면 국토법인 경개를 높이 토법인 경개는 국토법인 경개를 높이 객체개를 토법인 경개를 한다면 국토법인 경개는다\n\n국토법인 경개를 들여 국토법인 경개를 국토법인 경개를 높이 국토법인 경개를 한다면 국토법인 경개를 한다면 국토법인 토법인 경개를 높이 한다면 국토법인 경개를 높이 한다면 국토법인 경개를 한다면 국토법인 경개를 높이 한다면 국토법인 경개를 한다면 국토법인 경개를 한다'}


In [16]:
class GPTRM_custom(RewardModel):
    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [17]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
tokenizer = AutoTokenizer.from_pretrained(
    'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
    padding_side="right",
    model_max_length=512,
)

with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

In [18]:
train_data = total_data_ranking2chosen[100:]
eval_data = total_data_ranking2chosen[:100]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

20340
100


  0%|          | 0/20340 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [19]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
빈 인구가 몇이야
######################################################################
## chosen ##
죄송하지만, 저는 현재 시간과 위치를 파악할 수 없기 때문에 정확한 답변을 제공할 수 없습니다. 해당 정보는 지역의 인구 센서 혹은 행정기관에서 확인하실 수 있습니다.
######################################################################
## rejected ##
한다는 것을 한 같은 결 

"cih-jigh-puhs"

L-i-g-a-o-o-s-i-t-i-e-i-g-a-i-l-a-i-a-i-p-u-s


In [20]:
trainer = RewardModelTrainer(
    model=model,
    strategy=NaiveStrategy(),
    optim=torch.optim.Adam(model.parameters(), lr=5e-5),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    batch_size=8,
    max_epochs=1,
    early_stopping_rounds=None,
    bf16=False
)

In [21]:
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM')

Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/2543 [00:00<?, ?it/s]

[Epoch 1] dist: 9.9488, loss: 0.1233


In [22]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: -0.7


In [23]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: 5.4


In [24]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."
output_reward = inference_RM(input_text=input_text)

input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."
output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다.
reward score: 6.1
input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: 2.1


- 1epoch만 넘어가도 validation loss가 크게 증가해서 validation을 100개로 줄이고 early stop 사용 X
- BFloat16 타입도 사용하지 않음

- PPO

In [10]:
torch.cuda.empty_cache()
if model is not None:
    del model

In [11]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer
from copy import deepcopy

In [14]:
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(torch.cuda.current_device())
    tokenizer = AutoTokenizer.from_pretrained(
        'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
        padding_side="right",
        model_max_length=512
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

In [15]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [16]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [17]:
with open('./data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

len(list_prompt)

12000

In [18]:
list_data_dict[0:5]

[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'},
 {'prompt': '페르시아로부터 기원된 아랍요리의 특징이 뭐야'},
 {'prompt': '중화인민공화국의 기본 법률은 누가 제정 및 개정하는가?'}]

In [19]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [20]:
trainer.fit(
    list_prompt,
    num_episodes=3,
    max_timesteps=100,
    update_timesteps=300
)

actor.model.save_pretrained('models/output_3_PPO')

Episode [1/3]:   0%|          | 0/100 [00:00<?, ?it/s]

Episode [2/3]:   0%|          | 0/100 [00:00<?, ?it/s]

Episode [3/3]:   0%|          | 0/100 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/300 [00:00<?, ?it/s]

In [21]:
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained('models/output_3_PPO')
# model = AutoModelForCausalLM.from_pretrained('models/output_1_SFT')

In [22]:
print("\n" + "="*50)
print("PERFORMANCE EVALUATION (AFTER PPO)")
print("="*50)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id, # eos_token='</s>'의 ID 사용
    max_new_tokens=128,
    do_sample=True,
    top_k=50,
    temperature=1.0,
    early_stopping=True
)

evaluate_model(model, tokenizer, test_data)

Device set to use cuda:0



PERFORMANCE EVALUATION (AFTER PPO)
Generating predictions for evaluation...


  0%|          | 0/100 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Calculating scores...

Quantitative Evaluation Results
ROUGE Scores:
  rouge1: 11.11
  rouge2: 0.88
  rougeL: 11.18
  rougeLsum: 11.34

Qualitative Samples:

--- Sample 1 ---
✅ Prompt: 요 신발 얼마에요 사장님?
🎯 Ground Truth: '죄송하지만, 저는 AI 어시스턴트이며 제가 물건의 가격을 알 수는 없습니다. 상점에 가서 가격을 확인해보시는 것이 좋겠습니다!
🤖 Generated: '죄송합니다, 저는 가격 정보를 알 수 없습니다. 해당 제품의 판매처나 제조사에 문의하시는 것이 좋을 것 같습니다.
--------------------

--- Sample 2 ---
✅ Prompt: 돌아올거란 생각부터 하지 말걸
🎯 Ground Truth: 'As an AI language model, I am not capable of thinking, but I can provide a translation of the given text from Korean to English:\n\n"Don\'t even think that I\'ll come back."
🤖 Generated: '저는 AI 어시스턴트이기 때문에 돌봐드릴 수 없습니다. 하지만 언제든지 저와 대화할 준비가 되어있으시다면 언제든지 말씀해주세요!
--------------------

--- Sample 3 ---
✅ Prompt: 투표 당시 다른 후보들과의 일대일  대결에서 모두 지는 후보를 부르는 명칭은?
🎯 Ground Truth: '약점 후보(Antihero)
🤖 Generated: '제가 AI 어시스턴트이기 때문에, 정확한 답변을 드리기 위해서는 더 많은 정보가 필요합니다. 일반적으로, "투표"가 어떤 후보와의 일대일 대결을 의미하는지 알려주시면 더 정확한 답변을 드릴 수 있습니다.
--------------------


- 기존 KoChatGPT2: rougeL = 0
- SFT KoChatGPT2: rougeL = 7.62
- PPO KoChatGPT2: rougeL = 11.18
- SFT 모델과 육안상의 차이는 크게 없으나, 점수는 개선되었음

- PPO는 Ranking Model을 학습시키는 것과 PPO 자체의 과정 모두 잘 해야한다.
- RM, PPO 모두 1epoch이 넘어가면 loss가 증가해서 학습 자체를 많이 하지 않았다.
- SFT 데이터를 보면, ground truth에 영어 답변이 일부 존재한다. 이러한 데이터를 정제하고 새로운 데이터를 추가하면 더 좋아질 것 같다.